# AIFS Forecast from Brightband Initial Conditions

Runs ECMWF's [aifs-single-1.0](https://huggingface.co/ecmwf/aifs-single-1.0) model using
Brightband's [ECMWF IFS Initial Conditions](https://app.earthmover.io/marketplace/697162921880507a6587c31b)
from the Earthmover data marketplace, and writes the forecast back to Arraylake.

This notebook is an interactive, cell-by-cell version of `run_single_forecast`
in `main.py` — it imports the data loading and regridding functions from there,
so both entry points exercise exactly the same code path.

Run it on a Coiled GPU VM with `./run_notebook.sh` (the `--sync` flag makes
this repo directory, including `main.py`, available next to the notebook).

In [1]:
import datetime
import threading
import queue

import numpy as np
import pandas as pd
import torch

from arraylake import Client
from anemoi.inference.runners.simple import SimpleRunner
from anemoi.inference.outputs.printer import print_state

from main import (
    DEFAULT_IC_REPO,
    DEFAULT_TARGET_REPO,
    open_initial_conditions,
    fetch_initial_conditions,
    get_gpu_regridder,
    state_to_xarray,
    datetime_to_str,
)

In [2]:
client = Client()
client.login()

Successfully refreshed tokens! Token stored at /home/mambauser/.arraylake/token.json

╭───────────────────────────────────────────────── User Details ──────────────────────────────────────────────────╮
│ Name: Ryan Abernathey                                                                                           │
│ Email: ryan.abernathey@gmail.com                                                                                │
│ Id: 57df1a91-31e3-420d-8ad5-d8c82b11046b                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Open the initial conditions from the marketplace subscription

In [3]:
%%time
ds, ds_static = open_initial_conditions(DEFAULT_IC_REPO)
ds

CPU times: user 513 ms, sys: 59.6 ms, total: 573 ms
Wall time: 2.03 s


<xarray.Dataset> Size: 40GB
Dimensions:     (lead_time: 1, init_time: 99, latitude: 721, longitude: 1440,
                 level: 13)
Coordinates:
  * lead_time   (lead_time) timedelta64[s] 8B 00:00:00
    valid_time  (init_time, lead_time) datetime64[ns] 792B ...
  * init_time   (init_time) datetime64[ns] 792B 2026-06-13T18:00:00 ... 2026-...
  * latitude    (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
  * level       (level) float64 104B 1e+03 925.0 850.0 ... 150.0 100.0 50.0
Data variables: (12/26)
    d2m         (lead_time, init_time, latitude, longitude) float32 411MB ...
    hcc         (lead_time, init_time, latitude, longitude) float32 411MB ...
    mcc         (lead_time, init_time, latitude, longitude) float32 411MB ...
    q           (lead_time, init_time, level, latitude, longitude) float32 5GB ...
    sp          (lead_time, init_time, latitude, longitude) float32 411MB ...
    stl1        (lead_time, init_time, latitude, longitude) float32 411MB ...
    ...          ...
    swvl2       (lead_time, init_time, latitude, longitude) float32 411MB ...
    v10         (lead_time, init_time, latitude, longitude) float32 411MB ...
    z           (lead_time, init_time, level, latitude, longitude) float32 5GB ...
    v           (lead_time, init_time, level, latitude, longitude) float32 5GB ...
    v100        (lead_time, init_time, latitude, longitude) float32 411MB ...
    w           (lead_time, init_time, level, latitude, longitude) float32 5GB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts

In [4]:
# forecast from the most recent available analysis
date = pd.Timestamp(ds.init_time.values[-1]).to_pydatetime().replace(tzinfo=datetime.UTC)
print("Initial date is", date)

Initial date is 2026-07-08 06:00:00+00:00


## Fetch and regrid the input fields (0.25° ➞ N320, on the GPU)

In [6]:
input_regridder = get_gpu_regridder({"grid": (0.25, 0.25)}, {"grid": "N320"})
%time fields = fetch_initial_conditions(date, ds, ds_static, input_regridder)

CPU times: user 4.4 s, sys: 548 ms, total: 4.95 s
Wall time: 6.6 s


## Load the model

In [ ]:
checkpoint = {"huggingface": "ecmwf/aifs-single-1.0"}
runner = SimpleRunner(checkpoint, device="cuda")

In [ ]:
target_repo = client.get_or_create_repo(DEFAULT_TARGET_REPO)
target_session = target_repo.writable_session("main")

## Run the forecast

Outputs are regridded back to 0.25° on the GPU and written to Arraylake from a
background thread. The datasets stay numpy-backed (no dask) — writes happen
synchronously in the writer thread while the GPU works on the next step.

In [ ]:
output_regridder = get_gpu_regridder({"grid": "N320"}, {"grid": (0.25, 0.25)})

date_no_tz = date.replace(tzinfo=None)
input_state = dict(date=date_no_tz, fields=fields)

# we put data that we want to write into a queue
q = queue.Queue()
lock = threading.Lock()

def worker():
    while True:
        (ds_out, store, group_name, kwargs) = q.get()
        # lock is probably unncessary
        with lock:
            ds_out.to_zarr(
                store, group=group_name, zarr_format=3, consolidated=False, **kwargs
            )
        q.task_done()

# a separate thread for I/O to avoid blocking the main loop
threading.Thread(target=worker, daemon=True).start()

print("starting forecast loop")
kwargs = {"mode": "w"}

# clear GPU memory
torch.cuda.empty_cache()

# main forecast loop
for n, state in enumerate(runner.run(input_state=input_state, lead_time=48)):
    print_state(state)
    ds_out = state_to_xarray(state, regridder=output_regridder)
    group = datetime_to_str(date)
    if n > 0:
        kwargs = {"mode": "a", "append_dim": "valid_time"}
    q.put((ds_out, target_session.store, group, kwargs))

q.join()  # wait for all I/O tasks to finish

# clear GPU memory
torch.cuda.empty_cache()

In [ ]:
target_session.commit(f"Wrote a 48 hour forecast for {date}")